# Observatório de Pesquisa da Embrapa — embeddings na GPU

Calcula os embeddings (modelo `intfloat/multilingual-e5-base`) dos textos exportados pelo pipeline.

**Antes de começar:** menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)**.

Passos: execute as células em ordem (Shift+Enter).
1. Confere a GPU.
2. Você sobe as partes `textos_para_embeddings_parteXXdeYY.parquet` (geradas por `pipeline/py/exportar_textos.py`),
   uma por vez: a célula pede a próxima até receber todas.
3. Calcula os embeddings (~10–20 min para ~180 mil documentos numa T4).
4. Baixa `embeddings_colab.npy` e `embeddings_colab_ids.parquet` → coloque-os em
   `data/interim/embeddings/colab/` no projeto e rode `pipeline/py/embeddings.py`.


In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), "Sem GPU: Ambiente de execução → Alterar o tipo → GPU (T4)"
print(torch.cuda.get_device_name(0))

In [ ]:
!pip install -q -U sentence-transformers pyarrow

In [ ]:
from google.colab import files
import re, pandas as pd

# Envie as partes (uma ou várias por vez); a célula pede mais até receber todas.
partes = {}
while True:
    enviado = files.upload()
    partes.update(enviado)
    total = max(int(re.search(r"de(\d+)", n).group(1)) for n in partes)
    print(f"recebidas {len(partes)} de {total} partes")
    if len(partes) >= total:
        break

textos = pd.concat([pd.read_parquet(n) for n in sorted(partes)], ignore_index=True)
assert textos.doc_uid.is_unique
print(f"{len(textos):,} documentos".replace(",", "."))
textos.head(3)

In [ ]:
import time, numpy as np
from sentence_transformers import SentenceTransformer

MODELO = "intfloat/multilingual-e5-base"   # igual a pipeline/py/embeddings_cache.py
MAX_TOKENS = 512

modelo = SentenceTransformer(MODELO, device="cuda")
modelo.max_seq_length = MAX_TOKENS
modelo.half()                              # fp16: ~2x mais rápido na T4

t0 = time.time()
vetores = modelo.encode(textos.texto_modelo.tolist(), batch_size=128, normalize_embeddings=True,
                        show_progress_bar=True, convert_to_numpy=True)
print(f"{len(textos)/(time.time()-t0):.0f} docs/s · forma {vetores.shape}")
normas = np.linalg.norm(vetores.astype(np.float32), axis=1)
assert np.isfinite(vetores).all() and abs(normas.mean() - 1) < 0.01, "vetores inválidos"
print("ok: vetores normalizados")

In [ ]:
np.save("embeddings_colab.npy", vetores.astype(np.float16))
textos[["doc_uid", "hash"]].to_parquet("embeddings_colab_ids.parquet", index=False)
files.download("embeddings_colab_ids.parquet")
files.download("embeddings_colab.npy")     # ~280 MB; aguarde o download terminar